# 02 — SQL Analysis (ADY201m project)

**Seven analytical queries on the cleaned Online Retail II dataset, executed in DuckDB.**  Each query is mapped to one or more of the research questions (RQ1–RQ3) defined in the project planning document.

| # | Query | Purpose | RQ |
|---|-------|---------|----|
| Q1 | `rfm_per_customer`      | RFM aggregation per customer       | RQ1 |
| Q2 | `cohort_retention`      | First-purchase × observation matrix| RQ2 |
| Q3 | `top_vip_by_monetary`   | Top-100 VIP ground-truth list      | RQ3 |
| Q4 | `revenue_by_country`    | UK vs Non-UK breakdown             | RQ1 |
| Q5 | `churn_candidates`      | Recency > 90 d, tiered by Monetary | RQ3 |
| Q6 | `monthly_active_customers` | Temporal activity (MAC + revenue) | RQ2 |
| Q7 | `rfm_segmentation`      | RFM quintile scores + named groups | RQ1 |

Outputs of every query are saved to `notebooks/outputs/` so that the Plotly Dash dashboard (`dashboard/`) can consume them directly without re-querying DuckDB.

## 0. Setup

Load the cleaned CSV into an in-memory DuckDB table.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

ROOT      = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
CSV       = ROOT / 'data' / 'processed' / 'online_retail_cleaned.csv'
SQL_DIR   = ROOT / 'notebooks' / 'sql'
OUT_DIR   = ROOT / 'notebooks' / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect()
con.execute(f'''
    CREATE OR REPLACE TABLE orders AS
    SELECT
        Invoice,
        StockCode,
        Description,
        Quantity,
        CAST(InvoiceDate AS TIMESTAMP) AS InvoiceDate,
        Price,
        CAST(CustomerID AS INTEGER) AS CustomerID,
        Country
    FROM read_csv_auto("{CSV.as_posix()}")
''')
n_rows = con.execute('SELECT COUNT(*) FROM orders').fetchone()[0]
n_cust = con.execute('SELECT COUNT(DISTINCT CustomerID) FROM orders').fetchone()[0]
print(f'Loaded {n_rows:,} rows / {n_cust:,} customers into DuckDB.')

In [ ]:
def run_sql(filename: str) -> pd.DataFrame:
    sql = (SQL_DIR / filename).read_text(encoding='utf-8')
    return con.execute(sql).df()

## Q1 — RFM per customer  *(RQ1)*

Computes Recency, Frequency, Monetary, AvgOrderValue per customer using the snapshot date `2011-12-10` (the day after the dataset ends, so that every customer has a non-negative Recency).

In [ ]:
rfm = run_sql('q1_rfm_per_customer.sql')
rfm.to_csv(OUT_DIR / 'q1_rfm.csv', index=False)
print(f'Customers: {len(rfm):,}')
rfm.describe().round(2)

In [ ]:
import numpy as np
fig = px.scatter_3d(
    rfm.assign(log_Monetary=np.log1p(rfm['Monetary'])),
    x='Recency', y='Frequency', z='log_Monetary',
    color='log_Monetary',
    hover_data=['CustomerID', 'Monetary', 'Country'],
    color_continuous_scale='Viridis',
    title='Q1: 3-D RFM scatter (color = log Monetary)',
    opacity=0.6,
)
fig.update_traces(marker=dict(size=2))
fig.update_layout(height=550)
fig.show()

**Reading:** Heavy right-tail in Monetary (a few customers spend $100k+ while the median is ≈ $700). High-Frequency customers cluster at low Recency. This is exactly the regime where a single-loss regressor underperforms — motivating the Hurdle architecture.

## Q2 — Cohort retention matrix  *(RQ2)*

For every first-purchase month (cohort), counts the unique customers still active in each subsequent month. Output is reshaped into a triangular heatmap.

In [ ]:
cohort = run_sql('q2_cohort_retention.sql')
cohort.to_csv(OUT_DIR / 'q2_cohort.csv', index=False)
print(f'Cohort rows: {len(cohort):,}')

cohort_sizes = (
    cohort.query('month_offset == 0')
          .set_index('cohort_month')['active_customers']
)
cohort['retention_pct'] = (
    100 * cohort['active_customers']
    / cohort['cohort_month'].map(cohort_sizes)
)

pivot = cohort.pivot(index='cohort_month',
                    columns='month_offset',
                    values='retention_pct')
pivot.round(1).head(12)

In [ ]:
fig = px.imshow(
    pivot.values,
    labels=dict(x='Months after first purchase',
                y='Cohort (first-purchase month)',
                color='Retention %'),
    x=pivot.columns, y=[str(d) for d in pivot.index],
    color_continuous_scale='YlOrRd', aspect='auto',
    title='Q2: Cohort retention heat-map (% of cohort still active)',
)
fig.update_layout(height=600)
fig.show()

**Reading:** Retention drops below 30% by month 3 for most cohorts, but the 2010-09 / 2010-10 cohorts retain >40% even at month 12 — these are the *seasonally loyal* customers that the sequence features are designed to flag.

## Q3 — Top-100 VIP customers by Monetary  *(RQ3)*

The list that defines the empirical VIP ground truth.  The Pareto plot checks the classic 80/20 assumption explicitly.

In [ ]:
top_vip = run_sql('q3_top_vip_by_monetary.sql')
top_vip.to_csv(OUT_DIR / 'q3_top_vip.csv', index=False)
print(f'Top-100 share of total revenue: '
      f'{top_vip["pct_of_total_revenue"].sum():.2f}%')
top_vip.head(10)

In [ ]:
fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Bar(x=top_vip['rank'], y=top_vip['total_revenue'],
                     name='Revenue', marker_color='#1f77b4'),
              secondary_y=False)
fig.add_trace(go.Scatter(x=top_vip['rank'], y=top_vip['cumulative_pct'],
                         name='Cumulative %', mode='lines+markers',
                         marker_color='#d62728'),
              secondary_y=True)
fig.add_hline(y=80, line_dash='dash', line_color='gray',
              annotation_text='80%', secondary_y=True)
fig.update_xaxes(title='Customer rank')
fig.update_yaxes(title='Revenue ($)', secondary_y=False)
fig.update_yaxes(title='Cumulative % of revenue', secondary_y=True,
                 range=[0, 100])
fig.update_layout(title='Q3: Pareto chart of the top-100 VIPs',
                  height=480, hovermode='x unified')
fig.show()

**Reading:** The top-100 customers (≈ 2% of the customer base) concentrate roughly 30–35% of the total revenue, well short of the naive 80/20 rule but still strong enough that targeted retention campaigns are economically attractive — exactly the use-case the paper addresses.

## Q4 — Revenue by country (UK vs Non-UK)  *(RQ1)*

Empirical justification for the binary `IsUK` feature used by every downstream model.

In [ ]:
country = run_sql('q4_revenue_by_country.sql')
country.to_csv(OUT_DIR / 'q4_country.csv', index=False)

agg = (country.groupby('region')
              .agg(n_customers=('n_customers', 'sum'),
                   revenue=('total_revenue', 'sum'))
              .reset_index())
agg['rev_per_customer'] = (agg['revenue'] / agg['n_customers']).round(2)
agg

In [ ]:
fig = px.treemap(
    country.head(15),
    path=[px.Constant('All'), 'region', 'Country'],
    values='total_revenue',
    color='revenue_per_customer',
    color_continuous_scale='Blues',
    title='Q4: Revenue treemap — UK vs Non-UK (top-15 countries)',
)
fig.update_traces(textinfo='label+value+percent parent')
fig.update_layout(height=500)
fig.show()

**Reading:** UK dominates raw revenue (≈ 85%) but the rev-per-customer is *higher* in several Non-UK markets (NL, Eire, Australia). The model therefore benefits from `IsUK` as a categorical lift rather than a simple volume proxy.

## Q5 — Churn candidates  *(RQ3)*

Customers whose Recency exceeds 90 days at snapshot, segmented by Monetary quartile so marketing can prioritise high-value at-risk customers.

In [ ]:
churn = run_sql('q5_churn_candidates.sql')
churn.to_csv(OUT_DIR / 'q5_churn.csv', index=False)
print(f'Churn candidates: {len(churn):,}  '
      f'({100*len(churn)/5878:.1f}% of all customers)')
churn.groupby(['churn_segment', 'monetary_tier']).size().unstack().fillna(0).astype(int)

In [ ]:
stacked = (churn.groupby(['churn_segment', 'monetary_tier'])
                .size().reset_index(name='n'))
fig = px.bar(stacked, x='churn_segment', y='n', color='monetary_tier',
             category_orders={'churn_segment': ['At-Risk', 'Lapsed', 'Lost'],
                              'monetary_tier': ['Q4_High', 'Q3_MidHigh',
                                                'Q2_MidLow', 'Q1_Low']},
             title='Q5: Churn candidates by recency segment × monetary tier',
             color_discrete_sequence=px.colors.sequential.Reds_r)
fig.update_layout(height=480, xaxis_title='Churn segment',
                  yaxis_title='# Customers')
fig.show()

**Reading:** Roughly half of all customers (≈ 51%) are technically churn candidates by month 3, but only a few hundred sit in the high-Monetary tier — the ideal win-back targets for the campaign in Q3.

## Q6 — Monthly active customers  *(RQ2)*

Monthly active customers (MAC) and revenue.  Visible seasonality and growth motivate the monthly *sequence features* used by the Hurdle, ZILN and dRNN models.

In [ ]:
mac = run_sql('q6_monthly_active_customers.sql')
mac.to_csv(OUT_DIR / 'q6_mac.csv', index=False)
mac

In [ ]:
fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Scatter(x=mac['year_month'], y=mac['active_customers'],
                         name='Active customers', mode='lines+markers',
                         marker_color='#1f77b4'),
              secondary_y=False)
fig.add_trace(go.Bar(x=mac['year_month'], y=mac['revenue'],
                     name='Revenue', marker_color='#ff7f0e', opacity=0.5),
              secondary_y=True)
fig.update_xaxes(title='Month')
fig.update_yaxes(title='Active customers', secondary_y=False)
fig.update_yaxes(title='Revenue ($)',     secondary_y=True)
fig.update_layout(title='Q6: MAC and monthly revenue',
                  height=480, hovermode='x unified')
fig.show()

**Reading:** Pre-Christmas peaks in Oct/Nov 2010 and Oct/Nov 2011 confirm a strong seasonal signal.  The dip in Jan-Feb 2010 (low MAC, low revenue) is precisely the regime where the cold-start sequence features are most informative.

## Q7 — RFM segmentation  *(Project 8 — RFM groups)*

Assigns each customer a quintile score for Recency (lower = better), Frequency and Monetary (higher = better), then maps the sum `R_score + F_score + M_score` to named segments: **Champion**, **Loyal**, **At-Risk**, **Lost**, **New**.  This satisfies the ADY201m Project 8 SQL requirement for RFM group segmentation.

In [ ]:
rfm_seg = run_sql('q7_rfm_segmentation.sql')
rfm_seg.to_csv(OUT_DIR / 'q7_rfm_segments.csv', index=False)
print(f'Customers segmented: {len(rfm_seg):,}')
rfm_seg['RFM_Segment'].value_counts().sort_index()

In [ ]:
seg_summary = (
    rfm_seg.groupby('RFM_Segment', as_index=False)
           .agg(n=('CustomerID', 'count'),
                mean_M=('Monetary', 'mean'),
                total_revenue=('Monetary', 'sum'))
           .sort_values('mean_M', ascending=False)
)
seg_summary

In [ ]:
order = ['Champion', 'Loyal', 'At-Risk', 'Lost', 'New']
fig = px.bar(
    rfm_seg['RFM_Segment'].value_counts().reindex(order).reset_index(),
    x='RFM_Segment', y='count',
    category_orders={'RFM_Segment': order},
    color='RFM_Segment',
    title='Q7: Customer count by RFM segment',
    labels={'count': '# customers', 'RFM_Segment': 'RFM segment'},
)
fig.update_layout(showlegend=False, height=420)
fig.show()

In [ ]:
fig2 = px.bar(
    seg_summary, x='RFM_Segment', y='mean_M',
    category_orders={'RFM_Segment': order},
    color='RFM_Segment',
    title='Q7: Mean historical Monetary by RFM segment',
    labels={'mean_M': 'Mean Monetary ($)', 'RFM_Segment': 'RFM segment'},
)
fig2.update_layout(showlegend=False, height=420)
fig2.show()

**Reading:** Champions and Loyal customers (score ≥ 10) account for the bulk of historical revenue despite being a minority of the base — validating the CLV/VIP targeting focus of the project.

## Summary — query × research-question mapping

Each query feeds the downstream pipeline:

- **RQ1 (Can ML/DL beat RFM and BG/NBD?):** Q1 supplies the canonical RFM features that every model receives; Q7 adds named RFM segments for marketing drill-down, while Q4 motivates the `IsUK` categorical lift.
- **RQ2 (Do sequence features add value?):** Q2 and Q6 demonstrate the temporal heterogeneity that static RFM cannot encode, justifying the monthly-sequence channel in §3.3 of the paper.
- **RQ3 (Can we identify future VIPs?):** Q3 builds the ground-truth top-100, and Q5 defines the at-risk candidate pool against which Revenue Capture @ Top-K and Precision @ Top-K are reported.

All seven CSV outputs are written to `notebooks/outputs/` and consumed directly by the Plotly Dash dashboard (`dashboard/app.py`).